czjx

In [ ]:
import pandas as pd
import json




import re
from typing import Dict, Tuple

def extract_table_data(text: str) -> pd.DataFrame:
    """从文本中提取表格数据"""
    lines = text.strip().split('\n')
    data = []
    
    for line in lines:
        # 清理行数据，移除多余的空格和分割符
        line = line.strip()
        if line.startswith('|') and line.endswith('|'):
            # 分割单元格，移除首尾的 | 字符
            cells = [cell.strip() for cell in line[1:-1].split('|')]
            data.append(cells)
    
    return pd.DataFrame(data)

def normalize_value(value: str) -> str:
    """标准化数值，处理NA、空格等"""
    if pd.isna(value) or value == 'NA' or value == '':
        return 'NA'
    return str(value).strip().lower()

def compare_tables(excel_row: str, jsonl_md: str) -> Dict:
    """
    比较两个表格的内容并计算评估指标
    
    Args:
        excel_row: Excel表格文本
        jsonl_md: JSONL标记表格文本
        
    Returns:
        Dict: 包含比较结果和评估指标的字典
    """
    # 提取表格数据
    df_excel = extract_table_data(excel_row)
    df_jsonl = extract_table_data(jsonl_md)
    
    # 确保两个DataFrame有相同的形状
    max_rows = max(len(df_excel), len(df_jsonl))
    max_cols = max(len(df_excel.columns) if len(df_excel.columns) > 0 else 0, 
                   len(df_jsonl.columns) if len(df_jsonl.columns) > 0 else 0)
    
    # 重新索引以确保形状一致
    df_excel = df_excel.reindex(range(max_rows), fill_value='')
    df_jsonl = df_jsonl.reindex(range(max_rows), fill_value='')
    
    if max_cols > 0:
        df_excel = df_excel.reindex(columns=range(max_cols), fill_value='')
        df_jsonl = df_jsonl.reindex(columns=range(max_cols), fill_value='')
    
    comparisons = []
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    
    # 比较每个单元格
    for i in range(max_rows):
        row_comparison = []
        for j in range(max_cols):
            excel_val = normalize_value(df_excel.iloc[i, j] if i < len(df_excel) and j < len(df_excel.columns) else '')
            jsonl_val = normalize_value(df_jsonl.iloc[i, j] if i < len(df_jsonl) and j < len(df_jsonl.columns) else '')
            
            is_match = excel_val == jsonl_val
            
            # 计算分类指标（仅对非空且有意义的单元格）
            if excel_val != 'NA' and excel_val != '' and jsonl_val != 'NA' and jsonl_val != '':
                if is_match:
                    if excel_val not in ['', 'na']:  # 非空匹配视为真正例
                        true_positives += 1
                    else:  # 空值匹配视为真负例
                        true_negatives += 1
                else:
                    if excel_val not in ['', 'na'] and jsonl_val in ['', 'na']:
                        false_negatives += 1  # Excel有值但JSONL无值
                    elif excel_val in ['', 'na'] and jsonl_val not in ['', 'na']:
                        false_positives += 1  # Excel无值但JSONL有值
                    else:
                        # 两个都有值但不匹配，视为假正例和假负例
                        false_positives += 0.5
                        false_negatives += 0.5
            
            row_comparison.append({
                'row': i,
                'col': j,
                'excel': excel_val,
                'jsonl': jsonl_val,
                'match': is_match
            })
        comparisons.append(row_comparison)
    
    # 计算评估指标
    total_cells = sum(1 for i in range(max_rows) for j in range(max_cols) 
                     if normalize_value(df_excel.iloc[i, j]) != 'NA' and 
                        normalize_value(df_excel.iloc[i, j]) != '')
    
    # 准确率
    accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives) if (true_positives + true_negatives + false_positives + false_negatives) > 0 else 0
    
    # 精确度
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    
    # 召回率
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    # F1值
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'comparisons': comparisons,
        'metrics': {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'true_positives': true_positives,
            'false_positives': false_positives,
            'false_negatives': false_negatives,
            'true_negatives': true_negatives,
            'total_cells': total_cells
        },
        'excel_shape': (len(df_excel), len(df_excel.columns) if len(df_excel.columns) > 0 else 0),
        'jsonl_shape': (len(df_jsonl), len(df_jsonl.columns) if len(df_jsonl.columns) > 0 else 0)
    }

def print_comparison_results(results: Dict):
    """打印比较结果"""
    print("表格比较结果:")
    print("=" * 80)
    
    print(f"Excel表格形状: {results['excel_shape']}")
    print(f"JSONL表格形状: {results['jsonl_shape']}")
    print()
    
    print("详细比较:")
    print("-" * 80)
    
    matches = 0
    total = 0
    
    for row in results['comparisons']:
        for cell in row:
            if cell['excel'] != '' or cell['jsonl'] != '':
                status = "✓" if cell['match'] else "✗"
                print(f"位置({cell['row']},{cell['col']}): Excel='{cell['excel']}' | JSONL='{cell['jsonl']}' {status}")
                total += 1
                if cell['match']:
                    matches += 1
    
    print()
    print("评估指标:")
    print("-" * 80)
    metrics = results['metrics']
    print(f"准确率 (Accuracy): {metrics['accuracy']:.4f}")
    print(f"精确度 (Precision): {metrics['precision']:.4f}")
    print(f"召回率 (Recall): {metrics['recall']:.4f}")
    print(f"F1值 (F1-Score): {metrics['f1_score']:.4f}")
    print()
    print(f"真正例 (TP): {metrics['true_positives']}")
    print(f"假正例 (FP): {metrics['false_positives']}")
    print(f"假负例 (FN): {metrics['false_negatives']}")
    print(f"真负例 (TN): {metrics['true_negatives']}")
    print(f"总有效单元格: {metrics['total_cells']}")
    print(f"匹配单元格: {matches}/{total}")


# 逐行同步处理
df_excel = pd.read_excel('病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed copy.xlsx')
with open('deepseek-r1_results0.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if line.strip():
            # 读取 Excel 的对应行
            excel_row = df_excel.iloc[idx]
            jsonl_row = json.loads(line.strip())
            jsonl_md = jsonl_row['response']['body']['choices'][0]['message']['content']
            print(f"行号: {idx}")
            print(f"Excel: {excel_row['结构化表格']}")  # 替换为实际列名
            print(f"JSONL: {jsonl_md}")
        if idx == 1 :
            break

行号: 0
Excel: | 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|                             | 阴性        |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                              | NA             | NA        | 否                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 无             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | NA             | X1          | X1                     | 1                  | NA           |
| 髓外病灶(EM)                 | NA             | NA          | NA       

In [17]:

#deepseek qwenplus
import pandas as pd
import json
import re



            
# 配置参数



# 读取 Excel 数据

# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  
markdown_content =  """
下面是初诊基线评估的术语定义用来学习参考。

骨髓浸润与骨质破坏模式分类四类，分别为1.Minimal (normal appearing) 2.Focal lesions 3. Diffuse infiltration and bone destruction 4.Mixed (focal lesions on diffuse background)
骨髓浸润与骨质破坏模式 填写规范：Minimal/Focal lesions/Diffuse infiltration and bone destruction/Mixed  示例：Minimal

局灶性病灶绝对数目三类：0、1–3、>3
局灶性病灶绝对数目 填写规范：0/1–3/>3

骨折情况：需判断新增还是陈旧，骨折部位以及良恶性。
骨折情况 填写规范：文本（文本格式 新增/陈旧，部位文本，良性/恶性）示例：新增,T7，恶性

髓外病变：有 / 无（软组织/淋巴结/器官高代谢）
髓外病变 填写规范： 有/无

髓旁病变：有 / 无（自骨髓向外生长的软组织肿块）
髓旁病变 填写规范： 有/无

长骨浸润：仅指长骨，浸润指代谢弥漫性增高或有局限性病灶。有 / 无（股骨、肱骨等）
长骨浸润 填写规范： 有/无

骨骼手术证据：有 / 无（既往手术痕迹或金属植入）
骨骼手术证据 填写规范： 有/无


最终输出初诊基线评估结构化表格3行7列（待填写）如下：
| 骨髓浸润与骨质破坏模式   | 局灶性病灶绝对数目 |  骨折情况      |     髓外病变        |      髓旁病变      |    长骨浸润        |   骨骼手术证据   |
|------------------------|-----------------|----------------|------------------|--------------------|---------------------|----------------|
|  （待填写）             |  （待填写）      |     （待填写）   |   （待填写）      |    （待填写）      |  （待填写）          |（待填写）      |

真实示例：
| 骨髓浸润与骨质破坏模式   | 局灶性病灶绝对数目 | 骨折情况 | 髓外病变 | 髓旁病变 | 长骨浸润 | 骨骼手术证据 |
|------------------------|------------------|----------|----------|----------|-----------|--------------|
|  Diffuse infiltration and bone destruction    | 0               | 无       | 无       | 无       | 有        | 无           |

| 骨髓浸润与骨质破坏模式    | 局灶性病灶绝对数目 | 骨折情况                  | 髓外病变 | 髓旁病变 | 长骨浸润 | 骨骼手术证据 |
|---------------------|-------------------|---------------------------|----------|----------|----------|--------------|
| Focal lesions       | >3                 | 新增,左7肋及左10肋,恶性   | 无       | 有       | 有       | 无           |

"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
json_obj = []
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名


df_excel = pd.read_excel(r'病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed copy.xlsx')
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\qwen34bthinking结构化表格输出.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if idx > 121: 
            break
        excel_row = df_excel.iloc[idx]
        jsonl_row = json.loads(line.strip())
#for idx, row in df_excel.iterrows():
        # 拼接两列内容['body']['choices'][0]['message']['content']    病灶明细表：{excel_row['病灶明细表'] if not pd.isna(excel_row['病灶明细表']) else '无病灶'}
        user_content = f"PET-CT报告如下\n" + f"影像表现：{excel_row[col1]}  诊断结论：{excel_row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述" +"你是血液科医生"+f"PET-CT结构化表格：{jsonl_row['response']}  " + "初诊基线评估表格生成所需的多发性骨髓瘤PET-CT报告的结构化表格如上，请将初诊基线评估表格（待填写）部分填写完整，并严格按照其格式输出，最终仅输出结构化表格。"
        # 构建 JSON 对象
        json_obj.append({
            "custom_id": str(idx + 1),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "1",
                "messages":[
                    {"role": "user", "content": markdown_content + user_content}
                ],"temperature": 0, "top_p": 0.9}
        })
        
        # 写入文件（JSON Lines 格式）
        with open("qwen34bthinking初诊基线表输入prompt.jsonl", "w", encoding="utf-8") as f:    
            for item in json_obj:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


mrd

In [ ]:
"你是血液科医生"
""
下面是评估的术语定义用来学习参考。

MRD评估：
MRD阴性指的是必须同时满足下列所有影像学要求
骨髓所有原先受累区域骨髓信号/代谢恢复正常，无任何弥漫或局灶高代谢
局灶性病变完全消失：PET-CT 局灶计数 = 0，且无任何新发病灶
所有软组织肿瘤（paramedullary & extramedullary）完全消失
既往未受累区域不得出现任何新浸润或病灶

以上条件都满足
注意：MRD 阴性的“金标准”定义
• 必须 先满足骨髓 MRD 阴性（NGF 或 NGS，敏感度 ≥10⁻⁵）
• 且所有基线 PET-CT 阳性病灶完全消失，或 SUV 低于 纵隔血池或周围正常组织
• 建议 间隔 ≥1 年重复骨髓 + 影像均为阴性，方可称为“持续影像学 MRD 阴性

MRD阴性是否 填写规范： 是/否

最终输出MRD评估结构化表格3行7列（待填写）如下：
|  MRD阴性是否   | 
|----------------|
|  （待填写）     |

In [ ]:

#deepseek qwenplus
import pandas as pd
import json
import re




# 读取 Excel 数据

# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  
markdown_content =  """
下面是评估的术语定义用来学习参考。

MRD评估：
MRD阴性指的是必须同时满足下列所有影像学要求
骨髓所有原先受累区域骨髓信号/代谢恢复正常，无任何弥漫或局灶高代谢
局灶性病变完全消失：PET-CT 局灶计数 = 0，且无任何新发病灶
所有软组织肿瘤（paramedullary & extramedullary）完全消失
既往未受累区域不得出现任何新浸润或病灶

以上条件都满足
注意：MRD 阴性的“金标准”定义
• 必须 先满足骨髓 MRD 阴性（NGF 或 NGS，敏感度 ≥10⁻⁵）
• 且所有基线 PET-CT 阳性病灶完全消失，或 SUV 低于 纵隔血池或周围正常组织
• 建议 间隔 ≥1 年重复骨髓 + 影像均为阴性，方可称为“持续影像学 MRD 阴性

MRD阴性是否 填写规范： 是/否

最终输出MRD评估结构化表格3行1列（待填写）如下：
|  MRD阴性是否   | 
|----------------|
|  （待填写）     |
真实示例：
|  MRD阴性是否   | 
|----------------|
|     否         |
"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
json_obj = []

df_excel = pd.read_excel('病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed copy.xlsx')
with open('claudejiegouhuashuchu copy.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        excel_row = df_excel.iloc[idx+122]
        jsonl_row = json.loads(line.strip())
#for idx, row in df_excel.iterrows():
        # 拼接两列内容['body']['choices'][0]['message']['content']
        user_content = "你是血液科医生"+f"PET-CT结构化表格：{jsonl_row['response']}  病灶明细表：{excel_row['病灶明细表'] if not pd.isna(excel_row['病灶明细表']) else '无病灶'}" + "mrd评估表格生成所需的多发性骨髓瘤PET-CT报告的结构化表格和病灶明细表信息如上，请将MRD评估表格（待填写）部分填写完整，并严格按照其格式输出，最终仅输出结构化表格。"
        # 构建 JSON 对象
        json_obj.append({
            "custom_id": str(idx + 123),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "claude-haiku-4-5-20251001-thinking",
                "messages":[
                    {"role": "user", "content": markdown_content + user_content}
                ],"temperature": 0, "top_p": 0.9}
        })
        
        # 写入文件（JSON Lines 格式）
        with open("claudemrddm0shuru.jsonl", "w", encoding="utf-8") as f:    
            for item in json_obj:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


In [5]:

#deepseek qwenplus
import pandas as pd
import json
import re




# 读取 Excel 数据
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名
col3 = "病灶明细表"
col4 = "结构化表格"
# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  
markdown_content =  """我需要生成带思考过程的微调数据，我将提供提示词，输入信息，输出的正确答案，请根据这些生成带<think>思考过程包括在内</think>输出的正确答案
下面是原来的提示词：
下面是PET-CT报告的专业术语定义用来学习参考。

扫描区包括四肢，脊柱，颅骨。
在后面的填写规范中NA也表示不确定、不明，X4和NA都可指示多发

阳性信息  填写规范：阳性/阴性
PET阳性（PET-positive）的定义
满足以下任一条，即判定为阳性病灶：
1.局灶性骨病变：
  a.在基线或随访扫描中，出现 局灶性 FDG 摄取，且其 SUVmax ≥ 纵隔血池（mediastinal blood pool SUV）或 ≥ 周围正常骨髓组织；
  b.可伴或不伴 CT 可见的溶骨性破坏。
2.髓外病变（EM）或髓旁病灶(PM)：
  a.任何软组织或骨旁区域出现 新发或持续的高代谢灶（SUVmax 标准同上）。
3.弥漫性骨髓高代谢：
  a.骨髓弥漫性 FDG 摄取高于纵隔血池，且无其他可解释原因（如感染、骨折）
PET阴性（PET-negative）的定义
必须 同时满足 以下所有条件：
1.完全消失：
  a.与基线相比，所有先前的高代谢病灶（包括局灶性骨病变、EMD、弥漫性骨髓摄取） 完全消失；
2.或代谢降至背景以下：
  a.残余灶的SUVmax < 纵隔血池 SUV 或 < 周围正常骨髓组织；
3.无新发病灶：
  a.无新发局灶性或弥漫性高代谢灶；
4.无新骨质破坏：
  a.CT部分未见新发溶骨性病变。
对于初诊患者,PET-CT阴性指的是不满足PET-CT阳性任一条件。

骨髓/骨骼整体代谢活性 填写规范：是/否 数值 是/否
骨髓/骨骼整体代谢活性增高，主要看SUV值>肝脏代谢，通常以肝脏代谢活性为参照标准。需排除化疗后骨髓增生反应或生长因子使用导致的骨髓高代谢（可能表现为弥漫性摄取增高）。

骨质破坏 填写规范：有/无
骨质破坏指的是PET-CT中显示明确的溶骨性病变（如骨皮质缺损、溶骨性破坏、穿凿样改变）。


按 MM 专用 Deauville 5 级标准： 
1 分：病灶处完全无摄取，或肉眼不可辨。
2 分：可见摄取，但病灶 SUVmax ≤ 肝脏 SUVmax。
3 分：病灶 SUVmax ＞ 肝脏，但增幅 ≤ 10 %。
4 分：病灶 SUVmax ＞ 肝脏 +10 %，但未达 2 倍。
5 分：病灶 SUVmax ≥ 2× 肝脏，或出现任何新的高摄取灶。
放射性摄取增高指的是比周围骨髓组织高，或高于肝脏基础摄取值。

病变类别有局灶性病灶（FLs），髓外病灶(EM)，髓旁病灶(PM)，溶骨性病变(L)。
局灶性病灶（FLs）填写规范：S/SP/Ex-Sp X1/X2/X3/X4 X1/X2/X3/X4 数值 数值 
局灶性病变部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)，局灶性病变数量分为X1 (None) 、X2 (N =1 to 3)、 X3 (N =4 to 10)、 X4 (N >10),填写时应填写X1或X2或X3或X4。局灶性病变定义为在至少 2 个连续的 PET 切片上或大小大于5mm可见的 F-FDG 摄取增加的局灶区域，摄取增加指的是SUVmax值大于周围骨髓/骨骼组织或高于肝脏组织或大于2.5，需表现为局灶性（非弥漫性），即单个或多个离散的高摄取灶。需排除弥漫性摄取和生理学摄取（炎性摄取）。溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。
髓外病灶(EM) 填写规范： 文本 N或EN或N/EN X1/X2/X3/X4 X1/X2/X3/X4 数值
髓外病变（EM）指的是软组织/器官高摄取灶，髓外病变（EM）分为N(淋巴结)/EN(非淋巴结)。髓外病变淋巴结如果为高摄取灶，但诊断结论为随访、排除炎症、其他肿瘤性病变等则考虑髓外病变；对于非淋巴结组织，若诊断结论仅为随访考虑不是髓外病变。髓外病变（EM）部位填写时应填写N或EN或N/EN，当部位填写N/EN时数量可填写如X4/X2形式。
髓旁病灶(PM) 填写规范： 文本 X1/X2/X3/X4 X1/X2/X3/X4 数值 数值
髓旁病灶(PM)是指软组织肿块从骨髓生长到周围组织。髓外病变和髓旁病灶需要排除退行性病变、炎性病变以及非多发性骨髓瘤的肿瘤病变（影像表现和诊断结论作为依据）。
特别注意旁髓病变，临近骨且形成软组织肿块。

溶骨性病变(L) :溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。

SUVmax最大值 填写规范：数值 
SUVmax最大值指的是骨髓瘤病变，包括弥漫性病变，局灶性病变，髓外病变，髓旁病变。

骨折 填写规范：有/无 S/SP/Ex-Sp 新发/陈旧 是/否
骨折指CT病理性骨折，骨折部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)

骨骼系统外科手术证据 填写规范：有/无 文本 a/b/c/d
骨骼系统外科手术证据影像学报告中需明确标注 既往或近期骨骼系统手术痕迹，其类型分为：
  a.内固定植入物（如钢板、螺钉、髓内钉）
  b.椎体成形术/后凸成形术（骨水泥填充）
  c.骨移植或人工关节置换
  d.其他手术相关结构改变（如截骨术、刮除术痕迹）

所有数据如果是无法判断或无法获取填写规范即写NA

最终输出结构化表格16行6列（待填写）如下：
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|                             |  （待填写）      |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             |    （待填写）    | （待填写）   | （待填写）           |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             |  （待填写）      |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            |  （待填写）     |   （待填写）    |（待填写）           | （待填写）         |  （待填写）     |
| 髓外病灶(EM)                 |  （待填写）      |  （待填写）   |  （待填写）           |  （待填写）         |  （待填写）      |
| 髓旁病灶(PM)                |   （待填写）     |  （待填写）   | （待填写）            |  （待填写）           |  （待填写）     |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                |  （待填写）   |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | （待填写）     |  （待填写）   |   （待填写）        |  （待填写）        |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | （待填写）       |   （待填写）    |  （待填写）           |                      |             |

真实示例3例：
a
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

b
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

c
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
| ---------------------------- | -------------- | ---------- | --------------------- | ------------------ | ------------ |
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 无             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | NA             | X1         | 0                     | NA                 | NA           |
| 髓外病灶(EM)                 | NA             | NA         | NA                    | NA                 | NA           |
| 髓旁病灶(PM)                | NA             | NA         | NA                    | NA                 | NA           |
|                             | SUVmax最大值   |            |                       |                    |              |
| SUVmax最大值                | 2.7            |            |                       |                    |              |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |              |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                     |              |
"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
json_obj = []

df_excel = pd.read_excel(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\2\剩余数据.xlsx')

for idx, row in df_excel.iterrows():
        excel_row = df_excel.iloc[idx]
      #  jsonl_row = json.loads(line.strip())
#for idx, row in df_excel.iterrows():
        # 拼接两列内容['body']['choices'][0]['message']['content']
        user_content =f"输入信息如下：PET-CT报告\n" + f"影像表现：{row[col1]}  诊断结论：{row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述，同时我们将包含了具体原文内容的病灶明细表作为辅助输入如下(属于输入信息)"+f"病灶明细表：{row[col3]}"  + "输出的正确答案 如下:"+f"{row[col4]}"+"我需要生成带思考过程的微调数据，上面是提供的提示词，输入信息，输出的正确答案，请根据这些生成带<think>思考过程包括在内</think>输出的正确答案的微调数据" 
       # 构建 JSON 对象
        json_obj.append({
            "custom_id": str(idx),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "deepseek-v3.2-exp-thinking",
                "messages":[
                    {"role": "user", "content": markdown_content + user_content}
                ]}
        })
        
        # 写入文件（JSON Lines 格式）
        with open("微调数据.jsonl", "w", encoding="utf-8") as f:    
            for item in json_obj:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 
